# FIT3182 - Big data management and processing

## Introduction to Stream Joins

In this activity, we will learn and practice some of the useful APIs and functionalities in PySpark Structured Streaming that will help you performing join operation on streaming events.

***Instructions***

- You will be using Pyspark docker container provided in this unit.
- Read the markdown cells to understand the principle.
- Read the code base and comments carefully.
- After understanding the content, run the cell to check if the results is correct.
- Read carefully all the **Exercise** tasks below in each subheading. There may be some code blocks that **you need to complete** yourself.

**After this activity, you will**

- Be able to reason various stream joins
- Be able to implement stream join using PySpark API

# 1 Overview

A stream processing system continuously processes an **unbounded** sequence of events. Conceptually, it is an ever-growing table where new rows keep being appended. Spark Structured Streaming explicitly adopts this **stream as table** model as we've seen in previous tutorials.

## Stream Join

In real systems, data about the same real-world entity arrives via multiple streams:

- Orders stream with Payment stream (correlate transactions)
- Clicks stream with Impression stream (attribution)
- Sensor readings with device metadata (enrichment)

Stream joins combine two or more data streams (each represented a partial view of data) by matching records on common keys into a richer, derived dataset. Stream joins are fundamental to real-time analytics (fraud detection, user activity tracking etc.) because they provide a holistic view across multiple sources. Such derived views often must be kept in sync with their sources and that streaming can do this incrementally.

## Batch Join vs Stream Join

In the early chapters of this unit, we have learnt about batch processing through various operations in DBMS. Among those operations, join operation forms an important part of data pipelines, especially in distributed models like [MapReduce](https://static.googleusercontent.com/media/research.google.com/en//archive/mapreduce-osdi04.pdf). Since stream processing generalizes data pipelines to incremental processing of unbounded datasets, there is exactly the same need for joins on streams. However, the fact that new events can appear at any time on a stream makes joins on streams more challenging than in batch jobs.

The following illustrates the main difference of stream join against batch join

- Streams are unbounded. Without bounding, you must keep unbounded state to match future events which is infeasible in practical
- Out-of-order arrival. Arrival order across streams is not stable and join operation must handle out-of-order events or accept non-deterministic results

The need to store buffered records is why stream joins are stateful and why state management becomes the central engineering issue.

## Common Stream Join Problems

### Unbounded State Growth

With unbounded event stream, join operation is required to maintain unbounded state to match all potential future events which is impractical due to finite memory in a system. It often comes with symptoms like streaming query slows down over time and eventually runs out of memory.

Commonly, windowing operation based on event time is used as join operation to keep the state sustainable throughout the application livetime. The duration of the window can be adapted to suit the business requirements and the system capacity.

### Late Events

If a record arrives after you have already evicted the state of relevant partner record, the join would miss this match. In Spark, handling of such late arrival data can be achieved through [watermarking](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#handling-late-data-and-watermarking). However, this results in trade-off between correctness and latency. Increasing the lateness tolerance increases correctness but also increases latency and memory usage.

### Duplicates

Even with strong engine guarantees, upstream systems can produce duplicates, often as a result of retrying operation due to some system failures. In Spark, you can deduplicate records in data streams using a unique identifier in the events. For details, please refer to [here](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#streaming-deduplication).

# 2 Stream-Stream Joins

A stream-stream join pairs events from two unbounded streams based on key equity *and* a time relationship. Say you have a search feature on your website, and you want to detect recent trends in searched-for URLs. To get the data, you log an event containing the query and the results returned from the search and another event recording the click. To calculate the click-through rate, the events for the search action and the click action should be connected through the same session ID. 

Note that the click may never come if the user abandons their search and even if it comes, the time between the search and the click may be highly variable. In many cases, it might be a few seconds but it could be as long as days or weeks (depending on the habit and usage of the user). Due to variable network delays, click event may arrive before the search event, resulting in out-of-order arrival.

To implement this type of join, a stream processor needs to maintain state, i.e., all events that occured indexed by session ID. In Spark, the state is maintained through built-in [State Store](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#state-store), a versioned key-value store. Using micro-batch execution model in Spark, the join operation can be abstracted as follows

- Read data from streams
- For each partition in the state
    - Load previous state from disk
    - Update state with new records
    - Perform Join
    - Emit results
    - Apply watermark cleanup
    - Write updated state back to disk

Note that the disk storage is meant for fault tolerance and system durability. Spark relies on checkpointing and WAL (Write-Ahead Logging) for fault tolerance. Details can be found [here](https://learn.microsoft.com/en-us/azure/hdinsight/spark/apache-spark-streaming-exactly-once).

## Case Study

Suppose we have:

- A stream of click events (userId, productId, clickTime)
- A stream of purchase events (userId, productId, purchaseTime)

We want to find purchases that happened shortly after a click (e.g. within 10 minutes) for the same user and product. Conceptually, we have to do an inner join on (userId, productId) with a time filter of 10 minutes interval between click and purchase time.

For demonstration purpose, 2 set of csv files have been prepared, namely [click_events](data/click) and [purchase_events](data/purchase). The use of csv file is merely a simple emulation of streaming events in batches.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr
from pyspark.sql.utils import StreamingQueryException


def process_batch(batch_df, batch_id):
    record_count = batch_df.count()
    print(f"\n=== Batch ID: {batch_id} ===")
    print(f"Number of matched records: {record_count}")
    batch_df.show(record_count, truncate=False)


spark = SparkSession.builder.appName("StreamStreamJoin").getOrCreate()

root = "/home/student/stream-join/"
schema = "userId STRING, productId STRING, eventTime TIMESTAMP"

clickStream = (
    spark.readStream.format("csv")
    .schema(schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .load(root + "data/click")
    .withColumnRenamed("eventTime", "clickTime")
    .alias("clicks")
)

purchaseStream = (
    spark.readStream.format("csv")
    .schema(schema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .load(root + "data/purchase")
    .withColumnRenamed("eventTime", "purchaseTime")
    .alias("purchases")
)

joinedStream = clickStream.join(
    purchaseStream,
    expr("""
        clicks.userId = purchases.userId AND
        clicks.productId = purchases.productId AND
        purchases.purchaseTime BETWEEN clicks.clickTime AND clicks.clickTime + interval 10 minutes
    """),
)

try:
    query = (
        joinedStream.writeStream.outputMode("append")
        .foreachBatch(process_batch)
        .start()
    )

    query.awaitTermination()

except KeyboardInterrupt:
    print("Interrupted by CTRL-C. Stopped query")
except StreamingQueryException as exc:
    print(exc)
finally:
    query.stop()

## Exercise

In the above case study, there is an implementation issue with respect to the join operation. Can you spot it? Explain the issue and how would you fix it. Show the solution in the following cell.

# 3 Stream-Table Joins

A stream-table join combines a streaming input with a static lookup table. In Spark, the static side is usually a small reference table that does not change often. When a new record arrives in the stream, Spark joins it against the current version of the static table. Since only the new records are processed in each micro-batch, this join is stateless.

Suppose you have a set of user activity events and a database of user profiles. It is natural to think of the user activity events as a stream and perform the join operation with user profiles for information enrichment on a continuous basis in a stream processor. In this context, the input is a stream of activity events containing a user ID, and the output is a stream of activity events in which the user ID has been augmented with the profile information about the user. To perform this join, the user profile can be loaded into the stream processor so that it can be queried locally. In Spark, the user profile is loaded as a static dataframe which can be queried locally. However, Spark does not support all join types for stream-table joins. For details, refer [here](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#support-matrix-for-joins-in-streaming-queries).

## Case Study

Suppose we have:

- A stream of purchase events (userId, productId, eventTime)
- A database consisting of user profiles

We want to enrich the stream event with user profile which can help us to perform product analysis based on age group and country of the user. Conceptually, we have to perform an inner join on userId between the stream and the static table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.utils import StreamingQueryException


def process_batch(batch_df, batch_id):
    record_count = batch_df.count()
    print(f"\n=== Batch ID: {batch_id} ===")
    print(f"Number of matched records: {record_count}")
    batch_df.show(10, truncate=False)


spark = SparkSession.builder.appName("StreamTableJoin").getOrCreate()

root = "/home/student/stream-join/"
staticDataFrame = spark.read.csv(
    root + "data/user_profile.csv", header=True, inferSchema=True
)

eventSchema = "userId STRING, productId STRING, eventTime TIMESTAMP"
eventsStream = (
    spark.readStream.format("csv")
    .schema(eventSchema)
    .option("header", True)
    .option("maxFilesPerTrigger", 1)
    .load(root + "data/purchase")
    .withColumnRenamed("eventTime", "purchaseTime")
)

joinedDataFrame = eventsStream.join(staticDataFrame, on="userId", how="inner")

try:
    query = (
        joinedDataFrame.writeStream.outputMode("append")
        .foreachBatch(process_batch)
        .start()
    )
    query.awaitTermination()

except KeyboardInterrupt:
    print("Interrupted by CTRL-C. Stopped query")
except StreamingQueryException as exc:
    print(exc)
finally:
    query.stop()

## Exercise

In the above case study, the user profile is loaded as a static dataframe and used as a lookup table for joining operation. This setup is fine for most cases. However, when the user profile is constantly updated (i.e. new user is registered or existing user update their profile), what could be the potential problem with the current setup. How would you fix the issue? Demonstrate your solution in the following cell.

# 4 Table-Table Join

A table-table join is simply a normal batch join between 2 static DataFrames. In Spark, since there is no streaming aspect involved, the operation is run immediately. This join operation is the same as any Spark SQL join. This kind of join is often used for materialized view maintenance before feeding into a stream.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TableTableJoin").getOrCreate()
root = "/home/student/stream-join/"

df1 = spark.read.csv(
    root + "data/click/click_events_day_1.csv",
    header=True,
    inferSchema=True,
)

df2 = spark.read.csv(
    root + "data/purchase/purchase_events_day_1.csv",
    header=True,
    inferSchema=True,
)

# Join on a key column
joinedStatic = df1.join(df2, on=["userId", "productId"], how="inner")
joinedStatic.show()